# 01 - Gold / null calibration (CPU only, no GPU quota)

Set the accelerator to **None**.

Phase 2 verification never touches the model — `verify_task` takes the patch as a
plain string — so the whole grading path runs with no vLLM. That makes the two runs
which decide whether any later number means anything completely free.

- **gold**: feed each task its own `patch`. Every task *must* resolve. One that does
  not is broken: no agent can ever score it, so it is pure variance in the
  denominator of every future run.
- **null**: feed the empty patch. Every task *must* fail. One that resolves is a free
  point that lifts every config equally and dilutes the signal between them.

The surviving intersection is **clean-N**, and every number in this project is
reported on clean-N rather than on the raw 129.

This also produces the per-task Phase 2 wall times, which are what size the
concurrency probe in notebook 02.

In [ ]:
import os, subprocess, sys
from pathlib import Path

INPUT = Path('/kaggle/input')
print('attached inputs:', [p.name for p in sorted(INPUT.iterdir())] if INPUT.is_dir() else [])


def find_dir(*needles, must_contain='*.whl'):
    """Locate an attached dataset by name fragment, not a hardcoded path."""
    cands = []
    if INPUT.is_dir():
        for p in INPUT.rglob('*'):
            if p.is_dir() and any(n in p.name.lower() for n in needles):
                if not must_contain or any(p.glob(must_contain)):
                    cands.append(p)
    return sorted(cands, key=lambda q: len(str(q)))[0] if cands else None


WHEELHOUSE = find_dir('wheelhouse', must_contain='*.whl')
LAB = find_dir('swegemma-lab', must_contain='')
DATA = Path('/kaggle/input/competitions/gemma-4-developer-agent')
if not DATA.is_dir():
    DATA = find_dir('gemma-4-developer-agent', must_contain='tasks.jsonl') or DATA

print('wheelhouse:', WHEELHOUSE)
print('lab code  :', LAB)
print('data      :', DATA, DATA.is_dir())
assert WHEELHOUSE and LAB and DATA.is_dir(), 'attach wheelhouse + swegemma-lab + the competition data'
print('\ndata contents:', sorted(p.name for p in DATA.iterdir())[:12])

In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'

tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    if not (tmp / name).exists():
        os.symlink(w, tmp / name)

wheels = sorted(str(w) for w in tmp.glob('*.whl'))
print(f'installing {len(wheels)} wheels')
assert wheels
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', *wheels],
                   capture_output=True, text=True)
print('pip rc', r.returncode)
print((r.stderr or '')[-1500:] or '(no stderr)')

In [ ]:
# THE FIX for run 1, which measured 57% on gold and so failed its own gate.
#
# container_setup.install_test_dependencies returns immediately when the sandbox
# is SubprocessManager -- and subprocess is the only backend on Kaggle, since
# there is no Docker daemon. So the per-task sandbox never received the repos'
# test dependencies, pytest died at collection (35 tasks at exit code 2), and
# gold patches looked unresolvable.
#
# SubprocessManager sets system_site_packages=True, so whatever the notebook
# interpreter has is visible inside every per-task venv. Install there.
rc = subprocess.call([sys.executable, str(LAB / 'scripts' / 'stage_test_deps.py'),
                      '--data', str(DATA), '--lab', str(LAB)])
assert rc == 0, 'dependency staging failed; calibration would measure our setup, not the agent'

In [ ]:
# Check every harness import before doing any work. The harness is not
# installable locally, so a wrong module path is only discoverable here -- and
# without this gate it costs a whole kernel run to find. It already cost one:
# SessionTrace lives in adk_eval_core.tracing, not adk_eval_core.trace.
CAL = str(LAB / 'scripts' / 'calibrate.py')
rc = subprocess.call([sys.executable, CAL, '--preflight'])
assert rc == 0, 'preflight failed -- fix the import paths before spending a run'

In [ ]:
# Two tasks first. If the instrument is broken, find out in 3 minutes rather than 3 hours.
rc = subprocess.call([sys.executable, CAL, '--mode', 'gold', '--data-dir', str(DATA),
                      '--out', '/kaggle/working/smoke_gold', '--limit', '2'])
print('rc', rc)

**Stop here if the smoke test did not resolve 2/2.**

Run 1 resolved 74/129 (57%) on gold and was discarded: `install_test_dependencies`
is a no-op in subprocess mode, so no repo's test dependencies were ever present.
The staging cell above is the fix.

A gold patch that still fails to resolve means the harness, the snapshots or the
test dependencies are not wired up the way the grader wires them, and every
number produced afterwards would measure our setup rather than the agent. Read
the `test_output_tail` field in `smoke_gold/calibration.jsonl` before going on --
it distinguishes a collection error (exit 2, missing deps) from a genuine test
failure (exit 1).

In [ ]:
# Full gold run. Resumable: re-running skips tasks already in calibration.jsonl,
# so a session killed at the cap loses at most the task in flight.
rc = subprocess.call([sys.executable, CAL, '--mode', 'gold', '--data-dir', str(DATA),
                      '--out', '/kaggle/working/gold'])
print('rc', rc)

In [ ]:
rc = subprocess.call([sys.executable, CAL, '--mode', 'null', '--data-dir', str(DATA),
                      '--out', '/kaggle/working/null'])
print('rc', rc)

In [ ]:
rc = subprocess.call([sys.executable, CAL, '--reconcile', '/kaggle/working/gold',
                      '/kaggle/working/null', '--clean-out', '/kaggle/working/clean'])
print('rc', rc)

In [ ]:
# Phase 2 wall times per task -- these size the concurrency probe in notebook 02.
import json, statistics

rows = [json.loads(l) for l in Path('/kaggle/working/gold/calibration.jsonl')
        .read_text(encoding='utf-8').splitlines() if l.strip()]
secs = sorted(r['seconds'] for r in rows)
if secs:
    print(f'n={len(secs)}  p50={statistics.median(secs):.1f}s  '
          f'p90={secs[int(len(secs) * 0.9)]:.1f}s  max={secs[-1]:.1f}s  '
          f'total={sum(secs) / 60:.1f} min')
    by_repo = {}
    for r in rows:
        by_repo.setdefault(r['repo'], []).append(r['seconds'])
    for repo, v in sorted(by_repo.items(), key=lambda kv: -statistics.median(kv[1])):
        print(f'  {repo:22s} n={len(v):3d}  p50={statistics.median(v):6.1f}s')

In [ ]:
import shutil

OUT = Path('/kaggle/working/calibration'); OUT.mkdir(exist_ok=True)
for src in ('gold/calibration.jsonl', 'null/calibration.jsonl',
            'clean/clean_n.txt', 'clean/excluded.json'):
    p = Path('/kaggle/working') / src
    if p.is_file():
        shutil.copy(p, OUT / src.replace('/', '_'))
print(sorted(p.name for p in OUT.iterdir()))
print('\npull locally with:')
print('  kaggle kernels output ahmedsherif382/swegemma-gold-calibration -p results/calibration')
print('then commit clean_n.txt -- it is the denominator for every later number.')